In [1]:
from utils import load_disaster_dataset
from snorkel.augmentation import transformation_function, RandomPolicy, PandasTFApplier
import re

df_train, _, _, _ = load_disaster_dataset()

# 1. Write rules to change the text slightly
@transformation_function()
def tf_swap_disaster_synonyms(x):
    words = x.text.split()
    for i, w in enumerate(words):
        if w.lower() == "fire":
            words[i] = "blaze"
        elif w.lower() == "police":
            words[i] = "cops"
    x.text = " ".join(words)
    return x

@transformation_function()
def tf_remove_urls(x):
    x.text = re.sub(r'http\S+', '', x.text)
    return x

tfs = [tf_swap_disaster_synonyms, tf_remove_urls]

# 2. Apply the changes
policy = RandomPolicy(len(tfs), sequence_length=1, n_per_original=2, keep_original=True)

applier = PandasTFApplier(tfs, policy)
df_train_augmented = applier.apply(df=df_train)

print(f"Original size: {len(df_train)} | Augmented size: {len(df_train_augmented)}")

100%|██████████| 6090/6090 [00:01<00:00, 3832.97it/s]


Original size: 6090 | Augmented size: 18270
